In [1]:
import pandas as pd
import pyomo.environ as pyo
import time

### 1.Configuration & File Paths

In [4]:
COLLEGES_FILE = 'ties_colleges.csv'
SUBJECT_GROUPS_FILE = 'ties_groups.csv'
APPLICATIONS_FILE = 'ties_applications.csv'

# Prefixes for quota identifiers
COLLEGE_QUOTA_PREFIX = "C_"
SUBJECT_QUOTA_PREFIX = "S_"

### 2.Data loading

In [ ]:
def load_data():
    """Load raw data from CSV files."""
    df_colleges = pd.read_csv(COLLEGES_FILE)
    df_subject_groups = pd.read_csv(SUBJECT_GROUPS_FILE)
    df_apps = pd.read_csv(APPLICATIONS_FILE)
    return df_colleges, df_subject_groups, df_apps


def prepare_quota_system(df_colleges, df_subject_groups, df_apps):
    """
    Build the unified quota system and application data structures.

    Returns:
        - quotas_capacity: dict {quota_id: capacity}
        - college_to_quotas: dict {college_id: list_of_quota_ids}
        - E_list: list of (student_id, college_id) application pairs
        - rank_dict: dict {(student_id, college_id): rank}
        - score_dict: dict {(student_id, college_id): score}
        - student_choices: dict {student_id: list_of_college_ids}
        - unique_students: set of all student IDs
        - S_p_dict: dict {quota_id: sorted_list_of_unique_scores}
        - quota_applicants: dict {quota_id: set_of_student_ids}
    """
    
    # 2.1. Individual College Quotas

    quotas_capacity = {}
    college_to_quotas = {}

    for _, row in df_colleges.iterrows():
        cid = int(row['college_id'])
        q_col_id = f"{COLLEGE_QUOTA_PREFIX}{cid}"
        quotas_capacity[q_col_id] = int(row['capacity'])
        college_to_quotas[cid] = [q_col_id]

    # 2.2. Subject Group Quotas (Common quotas)
    for _, row in df_subject_groups.iterrows():
        q_sub_id = f"{SUBJECT_QUOTA_PREFIX}{int(row['group_id'])}"
        quotas_capacity[q_sub_id] = int(row['group_capacity'])

    # 2.3. Link each college to its subject group quota
    for _, row in df_colleges.iterrows():
        cid = int(row['college_id'])
        q_sub_id = f"{SUBJECT_QUOTA_PREFIX}{int(row['subject_id'])}"
        if q_sub_id in quotas_capacity:
            college_to_quotas[cid].append(q_sub_id)

    # 2.4. Process applications
    E_list = []
    rank_dict = {}
    score_dict = {}
    student_choices = {}
    unique_students = set()

    for _, row in df_apps.iterrows():
        i = int(row['student_id'])
        j = int(row['college_id'])
        rank = int(row['rank'])
        score = float(row['score'])   # same score for same subject (master ranking)

        unique_students.add(i)
        student_choices.setdefault(i, []).append(j)
        E_list.append((i, j))
        rank_dict[(i, j)] = rank
        score_dict[(i, j)] = score

    # 2.5. Extract scores per quota (S_p)
    quota_applicants = {p: set() for p in quotas_capacity}
    quota_scores_set = {p: set() for p in quotas_capacity}

    for (i, j) in E_list:
        score = score_dict[(i, j)]
        for p in college_to_quotas[j]:
            quota_applicants[p].add(i)
            quota_scores_set[p].add(score)

    S_p_dict = {
        p: sorted(list(scores))
        for p, scores in quota_scores_set.items()
        if scores
    }

    return (
        quotas_capacity,
        college_to_quotas,
        E_list,
        rank_dict,
        score_dict,
        student_choices,
        unique_students,
        S_p_dict,
        quota_applicants,
    )


### 3.Pyomo Model Construction

In [ ]:
def build_model(
    unique_students,
    df_colleges,
    E_list,
    rank_dict,
    score_dict,
    quotas_capacity,
    college_to_quotas,
    S_p_dict,
    quota_applicants,
    student_choices,
):
    """
    Build the COM‑Hungarian MIP model.
    Returns the Pyomo model and auxiliary data needed later.
    """
    K_VAL = max(rank_dict.values()) + 1

    # Precompute index sets for the model
    T_indices = [(p, s) for p in S_p_dict for s in S_p_dict[p]]
    E_p_indices = [
        (i, j, p)
        for (i, j) in E_list
        for p in college_to_quotas[j]
    ]
    D_indices = [
        (i, p)
        for p in S_p_dict
        for i in quota_applicants[p]
    ]

    model = pyo.ConcreteModel(name="COM_Hungarian")

    # ---------- Sets 
    model.A = pyo.Set(initialize=list(unique_students))
    model.C = pyo.Set(initialize=list(df_colleges['college_id']))
    model.E = pyo.Set(initialize=E_list, dimen=2)
    model.QuotaSet = pyo.Set(initialize=list(quotas_capacity.keys()))
    model.T_set = pyo.Set(initialize=T_indices, dimen=2)       # (quota, score)
    model.E_p_set = pyo.Set(initialize=E_p_indices, dimen=3)   # (student, college, quota)
    model.D_set = pyo.Set(initialize=D_indices, dimen=2)       # (student, quota)

    # ---------- Parameters 
    model.u_p = pyo.Param(model.QuotaSet, initialize=quotas_capacity)
    model.r = pyo.Param(model.E, initialize=rank_dict)
    model.s = pyo.Param(model.E, initialize=score_dict)

    # ---------- Variables (all binary) 
    model.x = pyo.Var(model.E, domain=pyo.Binary)          # x_{i,j}
    model.t = pyo.Var(model.T_set, domain=pyo.Binary)      # t_p^k
    model.e = pyo.Var(model.E_p_set, domain=pyo.Binary)    # e_{i,j}^p
    model.d = pyo.Var(model.D_set, domain=pyo.Binary)      # d_i^p

    # ---------- Objective (10): maximize student-optimal stable matching 
    def obj_rule(m):
        return sum((K_VAL - m.r[i, j]) * m.x[i, j] for i, j in m.E)
    model.Objective = pyo.Objective(rule=obj_rule, sense=pyo.maximize)

    # ---------- Constraint (1): student capacity 
    def student_cap_rule(m, i):
        return sum(m.x[i, j] for j in student_choices[i]) <= 1
    model.student_capacity = pyo.Constraint(model.A, rule=student_cap_rule)

    # ---------- Constraint (26): common quota capacity 
    def common_quota_cap_rule(m, p):
        relevant_apps = [(i, j) for (i, j) in E_list if p in college_to_quotas[j]]
        if not relevant_apps:
            return pyo.Constraint.Skip
        return sum(m.x[i, j] for i, j in relevant_apps) <= m.u_p[p]
    model.quota_capacity = pyo.Constraint(model.QuotaSet, rule=common_quota_cap_rule)

    # ---------- Constraint (32): admissibility
    def admissibility_rule(m, i, j, p):
        s_val = m.s[i, j]
        return m.x[i, j] <= m.t[p, s_val]
    model.admissibility = pyo.Constraint(model.E_p_set, rule=admissibility_rule)

    # ---------- Constraint (33): cutoff monotonicity 
    model.cutoff_monotonicity = pyo.ConstraintList()
    for p, scores in S_p_dict.items():
        for k in range(len(scores) - 1):
            model.cutoff_monotonicity.add(model.t[p, scores[k]] <= model.t[p, scores[k+1]])

    # ---------- Constraint (34): network envy-freeness
    def network_envy_free_rule(m, i, j):
        s_val = m.s[i, j]
        better_or_equal = [k for k in student_choices[i] if m.r[i, k] <= m.r[i, j]]
        sum_x_better = sum(m.x[i, k] for k in better_or_equal)

        # Sum of (1 - t_p^k) over all quotas p of college j
        sum_cutoffs = sum(1 - m.t[p, s_val] for p in college_to_quotas[j])

        return 1 <= sum_x_better + sum_cutoffs
    model.network_envy_free = pyo.Constraint(model.E, rule=network_envy_free_rule)

    # ---------- Constraints (36,37,38): identify margin candidates e_ij^p
    model.margin_candidate_e36 = pyo.ConstraintList()
    model.margin_candidate_e37 = pyo.ConstraintList()
    model.margin_candidate_e38 = pyo.ConstraintList()

    for (i, j, p) in E_p_indices:
        s_val = score_dict[(i, j)]
        scores = S_p_dict[p]
        k_idx = scores.index(s_val)

        # (36): e_ij^p = 1 only if student is NOT accepted to equal or better choice
        better_or_equal = [c for c in student_choices[i] if rank_dict[(i, c)] <= rank_dict[(i, j)]]
        model.margin_candidate_e36.add(
            model.e[i, j, p] <= 1 - sum(model.x[i, c] for c in better_or_equal)
        )

        # (37): student's score is exactly one step below the current cutoff
        if k_idx == len(scores) - 1:
            # highest score cannot be marginal
            model.margin_candidate_e37.add(model.e[i, j, p] <= 0)
        else:
            s_next = scores[k_idx + 1]
            model.margin_candidate_e37.add(
                model.e[i, j, p] <= model.t[p, s_next] - model.t[p, s_val]
            )

        # (38): student must pass cutoffs of all other quotas q of the same college
        for q in college_to_quotas[j]:
            if q != p:
                model.margin_candidate_e38.add(model.e[i, j, p] <= model.t[q, s_val])

    # ---------- Constraint (39): aggregate d_i^p from e_ij^p 
    def aggregate_d_rule(m, i, p):
        relevant_e = [
            (i, j, p)
            for j in student_choices[i]
            if p in college_to_quotas[j]
        ]
        if not relevant_e:
            return m.d[i, p] <= 0
        return m.d[i, p] <= sum(m.e[idx] for idx in relevant_e)
    model.aggregate_d = pyo.Constraint(model.D_set, rule=aggregate_d_rule)

    # ---------- Constraint (40): Hungarian non-wastefulness for quotas 
    def hungarian_quota_non_wasteful_rule(m, p):
        if not S_p_dict.get(p):
            return pyo.Constraint.Skip
        lowest_score = S_p_dict[p][0]

        relevant_apps = [(i, j) for (i, j) in E_list if p in college_to_quotas[j]]
        sum_x_p = sum(m.x[i, j] for i, j in relevant_apps)
        sum_d_p = sum(m.d[i, p] for i in quota_applicants[p])

        return (1 - m.t[p, lowest_score]) * (m.u_p[p] + 1) <= sum_x_p + sum_d_p
    model.hungarian_non_wasteful = pyo.Constraint(
        model.QuotaSet, rule=hungarian_quota_non_wasteful_rule
    )

    # Return the model and auxiliary data needed later
    return model, K_VAL, student_choices, S_p_dict, college_to_quotas, rank_dict, score_dict



### 4. Solve and Report

In [7]:
def solve_model(model, time_limit=None):
    """Solve the model using GLPK."""
    solver = pyo.SolverFactory('glpk')
    if time_limit:
        solver.options['tmlim'] = time_limit
    start = time.time()
    results = solver.solve(model, tee=True)
    end = time.time()
    return results, end - start


def print_results(model, results, solve_time, unique_students, K_VAL, S_p_dict,
                  quotas_capacity, rank_dict, score_dict, E_list):
    """Display solution summary and cutoff scores."""
    if results.solver.termination_condition != pyo.TerminationCondition.optimal:
        print(f"\nSolver failed. Status: {results.solver.termination_condition}")
        return

    print(f"\n*** Optimal Solution Found in {solve_time:.4f} seconds! ***")

    assignments = []
    assignment_tuples = []

    for (i, j) in E_list:
        if pyo.value(model.x[i, j]) > 0.5:
            assignments.append({
                'student_id': i,
                'college_id': j,
                'rank': rank_dict[(i, j)],
                'score': score_dict[(i, j)]
            })
            assignment_tuples.append((i, j))

    df_results = pd.DataFrame(assignments)
    df_results = df_results.sort_values('student_id').reset_index(drop=True)
    assignment_tuples.sort()

    print(f"Total Students Assigned: {len(df_results)} out of {len(unique_students)}")
    print(f"Solver Time: {solve_time:.4f} seconds")
    print(f"Solution Signature (Hash): {hash(tuple(assignment_tuples))}")

    print("\n--- Final Cutoff Scores ---")
    for p in S_p_dict.keys():
        final_cutoff = None
        for s in S_p_dict[p]:
            if pyo.value(model.t[p, s]) > 0.5:
                final_cutoff = s
                break

        q_type = "College" if p.startswith(COLLEGE_QUOTA_PREFIX) else "Subject"
        q_name = p.split("_")[1]

        if final_cutoff is not None:
            print(f"{q_type} {q_name:2} | Cutoff: {final_cutoff:.0f} | Capacity: {quotas_capacity[p]}")
        else:
            print(f"{q_type} {q_name:2} | No Cutoff (All rejected or 0 apps) | Capacity: {quotas_capacity[p]}")

### 5.Execute

In [8]:
print("Loading Data for COM-Hungarian Model (Full Dataset)...")
df_colleges, df_subject_groups, df_apps = load_data()

(
    quotas_capacity,
    college_to_quotas,
    E_list,
    rank_dict,
    score_dict,
    student_choices,
    unique_students,
    S_p_dict,
    quota_applicants,
) = prepare_quota_system(df_colleges, df_subject_groups, df_apps)

print(f"Data Ready: {len(unique_students)} Students, {len(df_colleges)} Colleges, "
      f"{len(df_subject_groups)} Common Quotas.")
print(f"Total Applications (E): {len(E_list)}")
print(f"Total Binary Cutoff Variables to create: {sum(len(scores) for scores in S_p_dict.values())}")

print("\nBuilding Pyomo Model... (This may take a minute for 1500 students)")
model, K_VAL, student_choices, S_p_dict, college_to_quotas, rank_dict, score_dict = build_model(
    unique_students,
    df_colleges,
    E_list,
    rank_dict,
    score_dict,
    quotas_capacity,
    college_to_quotas,
    S_p_dict,
    quota_applicants,
    student_choices,
)

print("\nSolving the model with GLPK... (This will take time due to NP-Hardness and dataset size)")
results, solve_time = solve_model(model)

print_results(
    model,
    results,
    solve_time,
    unique_students,
    K_VAL,
    S_p_dict,
    quotas_capacity,
    rank_dict,
    score_dict,
    E_list,
)

Loading Data for COM-Hungarian Model (Full Dataset)...
Data Ready: 15 Students, 5 Colleges, 4 Common Quotas.
Total Applications (E): 60
Total Binary Cutoff Variables to create: 92

Building Pyomo Model... (This may take a minute for 1500 students)

Solving the model with GLPK... (This will take time due to NP-Hardness and dataset size)
GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --write /tmp/tmp5pyjfune.glpk.raw --wglp /tmp/tmp9q1x3exl.glpk.glp --cpxlp
 /tmp/tmp2hqbizml.pyomo.lp
Reading problem data from '/tmp/tmp2hqbizml.pyomo.lp'...
/tmp/tmp2hqbizml.pyomo.lp:5089: warning: lower bound of variable 'x2' redefined
/tmp/tmp2hqbizml.pyomo.lp:5089: warning: upper bound of variable 'x2' redefined
768 rows, 384 columns, 2331 non-zeros
384 integer variables, all of which are binary
5473 lines were read
Writing problem data to '/tmp/tmp9q1x3exl.glpk.glp'...
4314 lines were written
GLPK Integer Optimizer 5.0
768 rows, 384 columns, 2331 non-zeros
384 integer varia